# cropy — 224 대비 256 입력 실험
기존 Drive ZIP 재사용. 같은 분할, ImageNet V2 초기화, seed 42, batch 32, 20 epoch, 일반 샘플링. 입력 크기만 256으로 변경합니다. 기존 cropy 체크포인트를 이어학습하지 않습니다.

GPU T4를 선택하고 1~4번을 순서대로 실행하세요. test는 보류합니다. 224 기준 val 91.85%, macro F1 0.9035. GPU/라이브러리가 바뀌면 해상도 이외의 차이가 생기므로 비교 시 기록을 확인하세요.

## 1. Drive 연결 및 원본 번들 검증·압축 해제

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, shutil, subprocess, sys, zipfile
from collections import Counter, defaultdict
import torch
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경 > GPU 선택'
BUNDLE = Path('/content/drive/MyDrive/cropy/cropy_group_baseline_bundle.zip')
assert BUNDLE.is_file(), f'업로드 경로 확인: {BUNDLE}'
EXPECTED_SHA256 = 'a9c21c21a31c6a7e18d9afd1d6fbb5d952b94df0787e5f6c59bd16acdba70725'
with BUNDLE.open('rb') as f:
    assert hashlib.file_digest(f, 'sha256').hexdigest() == EXPECTED_SHA256, '번들 해시 불일치'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
WORK = Path('/content') / ('cropy_group256_' + RUN_ID)
WORK.mkdir(exist_ok=False)
with zipfile.ZipFile(BUNDLE) as z:
    assert shutil.disk_usage(WORK).free > sum(i.file_size for i in z.infolist()) + 2_000_000_000
    for name in z.namelist():
        assert (WORK / name).resolve().is_relative_to(WORK.resolve()), '안전하지 않은 경로'
    z.extractall(WORK)
with (WORK / 'manifest.csv').open(encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))
workers, hashes = defaultdict(set), set()
for r in rows:
    with (WORK / r['bundle_path']).open('rb') as f:
        assert hashlib.file_digest(f, 'sha256').hexdigest() == r['sha256']
    assert r['sha256'] not in hashes
    hashes.add(r['sha256'])
    workers[r['filename_worker']].add(r['candidate_split'])
assert all(len(v) == 1 for v in workers.values())
assert Counter(r['candidate_split'] for r in rows) == dict(train=2728, val=785, test=722)
RESULTS = Path('/content/drive/MyDrive/cropy/results_256') / RUN_ID
print('검증 완료:', len(rows), Counter(r['candidate_split'] for r in rows))
print('GPU:', torch.cuda.get_device_name(0), '\n결과 경로:', RESULTS)

# ZIP의 224 전용 실행기를 대신할 최신 실행기를 노트북에 포함합니다.
(WORK / "train_group_baseline.py").write_text('"""ImageNet-only baseline for the audited worker split; no resume option."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torchvision\nfrom torch.utils.data import DataLoader\nfrom torchvision import datasets, models\n\nfrom train import get_transforms, train_epoch\n\n\ndef metrics(cm):\n    cm = np.asarray(cm, dtype=np.int64)\n    recall = np.divide(cm.diagonal(), cm.sum(1), out=np.zeros(len(cm)), where=cm.sum(1) > 0)\n    precision = np.divide(cm.diagonal(), cm.sum(0), out=np.zeros(len(cm)), where=cm.sum(0) > 0)\n    f1 = np.divide(2 * recall * precision, recall + precision,\n                   out=np.zeros(len(cm)), where=(recall + precision) > 0)\n    return dict(accuracy=float(cm.trace() / cm.sum()), macro_f1=float(f1.mean()),\n                recall=recall.tolist(), support=cm.sum(1).tolist(), confusion_matrix=cm.tolist())\n\n\n@torch.no_grad()\ndef evaluate(model, loader, device):\n    model.eval()\n    cm = np.zeros((len(loader.dataset.classes),) * 2, dtype=np.int64)\n    predictions = []\n    for images, labels in loader:\n        scores, preds = model(images.to(device)).softmax(1).max(1)\n        for actual, pred, score in zip(labels.tolist(), preds.cpu().tolist(), scores.cpu().tolist()):\n            cm[actual, pred] += 1\n            predictions.append(dict(actual=actual, predicted=pred, score=score))\n    for row, (path, _) in zip(predictions, loader.dataset.samples):\n        row[\'path\'] = str(Path(path).relative_to(loader.dataset.root))\n    return metrics(cm), predictions\n\n\ndef seed_worker(worker_id):\n    seed = torch.initial_seed() % 2**32\n    np.random.seed(seed)\n    random.seed(seed)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--data-dir\', type=Path, required=True)\n    p.add_argument(\'--output-dir\', type=Path, required=True)\n    p.add_argument(\'--epochs\', type=int, default=20)\n    p.add_argument(\'--batch-size\', type=int, default=32)\n    p.add_argument(\'--num-workers\', type=int, default=2)\n    p.add_argument(\'--seed\', type=int, default=42)\n    p.add_argument(\'--img-size\', type=int, choices=[224, 256], default=224)\n    p.add_argument(\'--evaluate-test\', action=\'store_true\', help=\'Use only after model/configuration selection is frozen\')\n    a = p.parse_args()\n    if a.evaluate_test:\n        device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        ckpt = torch.load(a.output_dir / \'best_model.pth\', map_location=\'cpu\', weights_only=True)\n        model = models.resnet50(weights=None)\n        model.fc = torch.nn.Linear(model.fc.in_features, len(ckpt[\'classes\']))\n        model.load_state_dict(ckpt[\'model_state_dict\'])\n        ds = datasets.ImageFolder(a.data_dir / \'test\', transform=get_transforms(ckpt[\'img_size\'])[1])\n        assert ds.classes == ckpt[\'classes\']\n        result, rows = evaluate(model.to(device), DataLoader(ds, batch_size=a.batch_size), device)\n        (a.output_dir / \'test_metrics.json\').write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n        (a.output_dir / \'test_predictions.json\').write_text(json.dumps(rows, ensure_ascii=False), encoding=\'utf-8\')\n        print(result)\n        return\n    if not torch.cuda.is_available():\n        raise SystemExit(\'Select a Colab GPU runtime before training.\')\n    if a.epochs < 1:\n        raise SystemExit(\'epochs must be positive\')\n    a.output_dir.mkdir(parents=True, exist_ok=False)\n    random.seed(a.seed)\n    np.random.seed(a.seed)\n    torch.manual_seed(a.seed)\n    torch.cuda.manual_seed_all(a.seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    train_tf, val_tf = get_transforms(a.img_size)\n    train_ds = datasets.ImageFolder(a.data_dir / \'train\', transform=train_tf)\n    val_ds = datasets.ImageFolder(a.data_dir / \'val\', transform=val_tf)\n    assert train_ds.classes == val_ds.classes and len(train_ds.classes) == 9\n    generator = torch.Generator().manual_seed(a.seed)\n    train_loader = DataLoader(train_ds, batch_size=a.batch_size, shuffle=True,\n                              num_workers=a.num_workers, worker_init_fn=seed_worker, generator=generator)\n    val_loader = DataLoader(val_ds, batch_size=a.batch_size, num_workers=a.num_workers,\n                            worker_init_fn=seed_worker)\n    config = dict(seed=a.seed, epochs=a.epochs, batch_size=a.batch_size, num_workers=a.num_workers,\n                  arch=\'resnet50\', img_size=a.img_size, initialization=\'ResNet50_Weights.IMAGENET1K_V2\',\n                  selection=\'highest val accuracy; earliest epoch on ties\', balanced_sampler=False,\n                  lr=1e-4, weight_decay=1e-4, label_smoothing=0.1,\n                  torch=str(torch.__version__), torchvision=str(torchvision.__version__),\n                  gpu=torch.cuda.get_device_name(0), classes=train_ds.classes,\n                  split_manifest_sha256=hashlib.sha256((a.data_dir.parent / \'manifest.csv\').read_bytes()).hexdigest())\n    (a.output_dir / \'config.json\').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)\n    model.fc = torch.nn.Linear(model.fc.in_features, 9)\n    model.to(\'cuda\')\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=a.epochs)\n    criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)\n    best = -1.0\n    for epoch in range(1, a.epochs + 1):\n        loss, accuracy = train_epoch(model, train_loader, criterion, optimizer, \'cuda\')\n        result, rows = evaluate(model, val_loader, \'cuda\')\n        record = dict(epoch=epoch, train_loss=loss, train_accuracy=accuracy / 100,\n                      lr=optimizer.param_groups[0][\'lr\'], val=result)\n        with (a.output_dir / \'history.jsonl\').open(\'a\', encoding=\'utf-8\') as f:\n            f.write(json.dumps(record) + \'\\n\')\n        print(f"Epoch {epoch}/{a.epochs}: train={accuracy:.2f}% val={result[\'accuracy\']:.2%} "\n              f"macro_f1={result[\'macro_f1\']:.4f} recall={result[\'recall\']}", flush=True)\n        if result[\'accuracy\'] > best:\n            best = result[\'accuracy\']\n            torch.save(dict(model_state_dict=model.state_dict(), classes=train_ds.classes,\n                            num_classes=9, arch=\'resnet50\', img_size=a.img_size, epoch=epoch,\n                            val_accuracy=best, config=config), a.output_dir / \'best_model.pth\')\n            (a.output_dir / \'val_metrics.json\').write_text(json.dumps(dict(epoch=epoch, **result), indent=2), encoding=\'utf-8\')\n            with (a.output_dir / \'val_predictions.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as f:\n                writer = csv.DictWriter(f, fieldnames=[\'actual\', \'predicted\', \'score\', \'path\'])\n                writer.writeheader()\n                writer.writerows(rows)\n        scheduler.step()\n    print(f\'Complete. Best val: {best:.2%}. Test not evaluated.\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding="utf-8")


## 2. ImageNet부터 256 학습
20 epoch / batch 32를 유지합니다. GPU 메모리 부족 시 오류를 공유하세요. batch를 바꾸면 입력 크기만 바꾸는 실험이 아니므로 224도 같은 batch로 재실험해야 합니다. 중단 후 자동 이어학습은 지원하지 않습니다. 결과는 Drive results_256에 보존됩니다.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 32
command = [sys.executable, '-u', str(WORK / 'train_group_baseline.py'),
           '--data-dir', str(WORK / 'dataset'), '--output-dir', str(RESULTS),
           '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--seed', '42', '--img-size', '256']
RESULTS.parent.mkdir(parents=True, exist_ok=True)
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
    for line in proc.stdout:
        print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError('학습 실패. 위 오류를 확인하세요.')
shutil.copy2(WORK / 'manifest.csv', RESULTS / 'manifest.csv')


## 3. 검증 결과 확인
클래스별 수치는 재현율입니다. test는 아직 평가하지 않습니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
config = json.loads((RESULTS / 'config.json').read_text())
result = json.loads((RESULTS / 'val_metrics.json').read_text())
print('선택 epoch:', result['epoch'], 'val:', f"{result['accuracy']:.2%}", 'macro F1:', result['macro_f1'])
display(pd.DataFrame({'class': config['classes'], 'recall': result['recall'], 'count': result['support']}))
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(result['confusion_matrix'], cmap='Greens')
for i, row in enumerate(result['confusion_matrix']):
    for j, count in enumerate(row):
        ax.text(j, i, str(count), ha='center', va='center')
ax.set(xlabel='Predicted class ID', ylabel='Actual class ID', title='Worker split / val', xticks=range(9), yticks=range(9))
fig.tight_layout()
fig.savefig(RESULTS / 'val_confusion.png')
plt.show()
plt.close(fig)

baseline = {'epoch': 20, 'accuracy': 0.9184713375796179, 'macro_f1': 0.903525385131823, 'recall': [0.9090909090909091, 0.9405940594059405, 0.8160919540229885, 0.6938775510204082, 0.9726027397260274, 0.7647058823529411, 1.0, 1.0, 1.0], 'support': [99, 101, 87, 49, 73, 68, 109, 87, 112], 'confusion_matrix': [[90, 2, 0, 0, 0, 0, 7, 0, 0], [1, 95, 0, 0, 0, 0, 5, 0, 0], [0, 0, 71, 14, 0, 0, 0, 2, 0], [0, 0, 14, 34, 0, 0, 1, 0, 0], [0, 0, 0, 0, 71, 2, 0, 0, 0], [0, 0, 0, 1, 1, 52, 0, 0, 14], [0, 0, 0, 0, 0, 0, 109, 0, 0], [0, 0, 0, 0, 0, 0, 0, 87, 0], [0, 0, 0, 0, 0, 0, 0, 0, 112]]}
display(pd.DataFrame({
    'class': config['classes'], 'recall_224': baseline['recall'],
    'recall_256': result['recall'],
    'delta_pp': [100*(b-a) for a,b in zip(baseline['recall'], result['recall'])],
}))
print('accuracy difference (percentage points):', 100*(result['accuracy']-baseline['accuracy']))
print('macro F1 difference:', result['macro_f1']-baseline['macro_f1'])


## 4. 結果 ZIP 다운로드 / 런타임 초기화 후 복구
이 셀은 앞 셀의 변수 없이도 실행할 수 있습니다. 학습이 끝난 뒤 사용하세요. 폴더가 여러 개이면 목록의 번호를 입력합니다. epoch 기록이 20개 미만이면 미완료임을 표시합니다.

In [ ]:
import shutil, json
from pathlib import Path
from google.colab import drive, files
drive.mount('/content/drive')
results_root = Path('/content/drive/MyDrive/cropy/results_256')
folders = sorted(p for p in results_root.glob('*') if p.is_dir() and (p/'config.json').is_file())
if not folders:
    raise RuntimeError('results_256에 결과가 없습니다. 2번 학습 실행 여부와 Drive 계정을 확인하세요.')
for i,p in enumerate(folders): print(f'[{i}] {p.name}')
index = 0 if len(folders)==1 else int(input('다운로드할 폴더 번호: '))
if not 0 <= index < len(folders): raise ValueError('목록 번호를 입력하세요.')
selected = folders[index]
config_saved = json.loads((selected/'config.json').read_text())
history_path = selected/'history.jsonl'
history_saved = [json.loads(s) for s in history_path.read_text().splitlines() if s.strip()] if history_path.exists() else []
print('선택:', selected, '기록된 epoch:', len(history_saved), '/', config_saved['epochs'])
if len(history_saved) < config_saved['epochs']:
    print('주의: 학습 미완료 기록입니다. 실행 중이라면 완료 후 다운로드하세요.')
result_zip = shutil.make_archive(str(Path('/content') / ('cropy_group256_results_' + selected.name)), 'zip', selected)
shutil.copy2(result_zip, results_root / Path(result_zip).name)
files.download(result_zip)
